In [1]:
alias_url = {
    'Brazil'  : 'dados.gov.br',
    'Dominican Republic' : 'www.datos.gob.do',
}

In [2]:
#alias = 'Dominican Republic'
#query     = 'estudio'

In [3]:
alias = 'Brazil'
query = 'IBGE Censo'

In [4]:
hyperlink = 'https://'+alias_url[alias]

# Módulos e pacotes

## Instalação

In [5]:
!pip install unidecode

In [6]:
!pip install pymupdf

In [7]:
!pip install lxml

In [8]:
!pip install pandas

In [9]:
!pip install ipywidgets

## Importação

In [11]:
import re
import fitz
import requests
import pandas

from unidecode import unidecode

from itertools import chain
from lxml.etree import HTML

import IPython
import ipywidgets
#import google.colab

# Funções

## auxiliares

#### lambdas

##### strip

In [12]:
strip_ = lambda T: set( t.strip() for t in T if t.strip() )
strip  = lambda T: '\n'.join( strip_( T ) ).strip()

##### plain

In [13]:
plain  = lambda I: list( chain.from_iterable( I ) )

## get_url_html

In [14]:
def get_url_html(url):
    requested         = requests.request('GET', url)
    html              = HTML(requested.text)    
    html.nsmap['url'] = url

    return html


## xpath

### contains_clause

In [15]:
def contains_clause(dic, operator='OR'):
    predicados = ["contains(@{k},'{v}')".format(k=k,v=v) for k,v in dic.items()]

    return operator.join(predicados)


### compose_xpath

In [16]:

def compose_xpath(element,attribute,content):
    dictionary = contains_clause({attribute:content})

    str_dict = {
        'element' : element,
        'dictionary' : dictionary 
    }

    return "{element}[{dictionary}]".format(**str_dict)


### get_xnodes

In [17]:

def get_xnodes(html, element, attribute='class', content='', complement=''):
    if complement:
        xnode = html.xpath(compose_xpath(element,attribute,content) + complement)
    else:
        xnode = html.xpath(compose_xpath(element,attribute,content) )
    return xnode if xnode else None


### get_xnode

In [18]:

def get_xnode(html, element, attribute='class', content='', complement=''):
    xnodes = get_xnodes(html, element, attribute, content, complement)
    return xnodes[0] if xnodes else None


## datasets

### get_html_page_count

In [19]:
 
def get_html_page_count(html):
    xnode = get_xnode( html, '//div', 'class', 'pagination' ) 
    
    if xnode is None:
        return 1
    
    pdiv = xnode.xpath( '//li/a/text()' )
    
    def pmax(pdiv):
        V = [1]
        for p in pdiv:
            try:
                v = int(p)
                V.append(v)       
            except:
                continue
        return max(V)    
    return pmax(pdiv)

### get_query_page_count

In [20]:

def get_query_page_count(hyperlink,query):
    html = get_url_html( hyperlink + '/dataset?q=%s' % query)
    return get_html_page_count( html )


### get_html_items_href






In [21]:

def get_html_items_href(html):
    xnodes = get_xnodes( html, './/h3', 'class', 'dataset-heading' , '//a/@href' ) 
    return xnodes


### get_html_dataset

In [22]:

def get_html_dataset(html):    
    article_html = get_xnode(html , '//article')
    
    dataset = dict()
    
    dataset['organization'] = ' '.join(set([x.strip() for x in get_xnodes(html, './/div','class','module-content','//h1//text()')]))
    dataset['title']        = strip( get_xnodes( article_html , 'div/h1/text()') ) 
    dataset['key']          = re.sub('[^\w]','_',unidecode(dataset['title']).lower())
    z = get_xnodes(article_html,'//div','class','notes','//p/text()')
    dataset['text'] = strip(z) if z else ''
    dataset['url']          = get_xnodes( article_html, '//ol', 'class', 'breadcrumb', '//li/a/@href')

    return dataset


### get_html_dataset_license

In [23]:

def get_html_dataset_license(html,hyperlink='https://dados.gov.br'):
    xnode = get_xnode( html, '//section','class','license') 

    if not xnode: return dict(title='NA',href='',text='',logo='')

    license_title = xnode.xpath( './/@title' )
    license_href  = xnode.xpath( './/@href')
    license_text  = [ text.strip() for text in xnode.xpath( './/text()' ) if text.strip() ]
    license_logo  = [ hyperlink + p for p in xnode.xpath( './/img//@src' ) ]
    
    license_dict          = dict()
    license_dict['title'] = license_title
    license_dict['href']  = license_href
    license_dict['text']  = license_text
    license_dict['logo']  = license_logo

    return license_dict


### get_html_dataset_resources_href

In [24]:

def get_html_dataset_resources_href(html):
    return get_xnodes(html,'//a','class','heading','/@href')


### get_html_resource_info





In [25]:
def get_html_resource_info(html):
    
    ths     = get_xnodes( html, '//table', 'class', 'table-condensed','/tbody//th/text()')
    headers = [re.sub('[^\w]','_',unidecode(th.lower())) for th in ths]
    tds     = get_xnodes( html, '//table', 'class', 'table-condensed','/tbody//td')
    data    = [strip(get_xnodes(td, './/text()')).strip() for td in tds]

    content = get_xnode( html, '//div','class','module-content')


    info_dict        = {h:d for h,d in zip(headers,data)}
    info_dict['resource_url'] = str(get_xnode( html, '//a','class','resource-url-analytic','/@href'))
    
    info_dict['resource_title'] = str(get_xnode( content, './/h1', 'class', 'page-heading','//text()'))
    info_dict['resource_description'] = str(get_xnode( content, './/div', 'class', 'prose notes','//text()'))

    return info_dict

### auxiliares

#### \_\_dir_repr\_\_

In [26]:

def __dir_repr__(variable=dict,pattern=''):
    dir_type_dict = dict()
    
    for attribute in dir(variable):
        if not re.findall(pattern, attribute): continue
        value = getattr(variable,attribute)
        typename = type(value).__name__
        
        if typename not in dir_type_dict:
            dir_type_dict[typename] = [(attribute,value)]
        else:
            dir_type_dict[typename].append((attribute,value))
        
    for key, items in dir_type_dict.items():
        print(key)
        print()
        for attribute,value in items:
            print('\t',attribute)
        print()
        #print('{: <30s}{:<20s}'.format(typename, attribute))


#### show_url

In [27]:

def show_url( url='https://'+hyperlink ):
    dadosgov_frame = IPython.display.IFrame(src=url,width='100%',height='500px')
    display(dadosgov_frame)
    return


#### query_site

In [28]:

def query_site( query=''):
    query = query.lower()
    query = unidecode.unidecode(query)
    query = re.sub('[^a-z\s\'\"]','',query)
    query = re.sub('\s+','+',query)
    
    url = hyperlink='/dataset?q=%s' % query
    print(url)
    dadosgov_frame = IPython.display.IFrame(src=url,width='100%',height='500px')
    display(dadosgov_frame)
    return dadosgov_frame



#### lambdas

##### get_query_page_url

In [29]:
get_query_page_url   = lambda h, q, p : h + '/dataset' + '?q=' + q + ( ( '&page=' + str(p) ) if ( p > 1 ) else '' )

##### get_query_page_html

In [30]:
get_query_page_html  = lambda h, q, p : get_url_html( get_query_page_url( h, q, p ) )


##### get_query_page_items

In [31]:
get_query_page_items = lambda h, q, p : get_html_items_href( get_query_page_html( h, q, p ) )


##### get_query_items_href

In [32]:
get_query_items_href = lambda h, q    : plain([ get_query_page_items( h, q, p+1 ) for p in range( get_query_page_count( h, q ) ) ])


# Leitura

In [36]:
query   = re.sub('[^\w]',' ',unidecode(query))
query   = re.sub('\s+'  ,' ',query)
query   = re.sub('\s'   ,'+',query)

query = 'UFRN discentes'

In [37]:
query_url = get_query_page_url(hyperlink,query,1)

In [39]:
print(query_url)
show_url(query_url)

https://dados.gov.br/dataset?q=UFRN discentes


## visualização

In [40]:
resources = []

In [41]:
items_href = []
page_count = get_query_page_count( hyperlink , query )
page_count

1

In [42]:
for page in range( page_count ):
    print('page {page} of {page_count}'.format(page_count=page_count,page=page+1) )
    html = get_query_page_html( hyperlink, query, page+1 )
    html_items = get_html_items_href(html)
    tuple(map(lambda item: print('\t',hyperlink+item), html_items));
    items_href += html_items

page 1 of 1
	 https://dados.gov.br/dataset/discentes
	 https://dados.gov.br/dataset/dados-complementares-de-discentes
	 https://dados.gov.br/dataset/dados-socio-economicos-de-discentes


In [43]:
ih = 0
 
print(hyperlink + items_href[ih])
show_url(hyperlink + items_href[ih])

https://dados.gov.br/dataset/discentes


## arquivos

In [44]:
import time

In [45]:
import pandas
resources = []
for item_href in items_href:
    item_url = hyperlink + item_href
 
    print('\n\n\t', item_url, end='\n\n')
    dataset_html   = get_url_html( item_url )
 
    dataset        = get_html_dataset( dataset_html )
    license        = get_html_dataset_license ( dataset_html )
    resources_href = get_html_dataset_resources_href( dataset_html )
 
    for resource_href in resources_href:
        resource_url  = hyperlink + resource_href 
 
        resource_html = get_url_html( resource_url )  
        resource_dict = get_html_resource_info(resource_html)
 
        a = 'http://landpage-h.cgu.gov.br/dadosabertos/index.php?url='
        resource_dict['resource_url_out'] = resource_dict['resource_url'].replace(a,'') if resource_dict['resource_url'] else None        

        resource_dict.update({('dataset_'+k):v for k,v in dataset.items()})
        resource_dict.update({('license_'+k):v for k,v in license.items()})
 
        resource_dict['havested'] = time.ctime()
        
        print('\t\t',resource_dict['resource_url_out'], end='\n')
 
        resources.append( resource_dict )



	 https://dados.gov.br/dataset/discentes



<ipython-input-23-b0cbb787167f>:4: FutureWarning: The behavior of this method will change in future versions. Use specific 'len(elem)' or 'elem is not None' test instead.
  if not xnode: return dict(title='NA',href='',text='',logo='')



		 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/a55aef81-e094-4267-8643-f283524e3dd7/download/discentes-2019.csv
		 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/146b749b-b9d0-49b2-b114-ac6cc82a4051/download/discentes-2018.csv
		 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/dc732572-a51a-4d4a-a39d-2db37cbe5382/download/discentes-2017.csv
		 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/7d2fa5b3-743f-465f-8450-91719b34a002/download/discentes-2016.csv
		 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/e2b5b843-4f58-497e-8979-44daf8df8f94/download/discentes-2015.csv
		 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/6c23a430-9a7c-4d0f-9602-1d5d97d40e6a/download/discentes-2014.csv
		 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/dba208c2-822f-4e26-adc3-b61d4cb110b6/download/discentes-2

# Dados

In [52]:
df_datagov = pandas.DataFrame(resources)

In [53]:
df_datagov = df_datagov[sorted(df_datagov.keys())]

In [54]:
df_datagov.sample(3).T

,5,44,28
created,há mais de 4 anos,há mais de 4 anos,há 4 meses
criado,27/Outubro/2017,23/Novembro/2017,18/Agosto/2021
dataset_key,discentes,dados_socio_economicos_de_discentes,discentes
dataset_organization,Discentes,Dados Sócio-Econômicos de Discentes,Discentes
dataset_text,Relação dos discentes da UFRN.,Apresenta os dados sócio-econômicos dos discen...,Relação dos discentes da UFRN.
dataset_title,Discentes,Dados Sócio-Econômicos de Discentes,Discentes
dataset_url,"[/, /organization, /organization/universidade-...","[/, /organization, /organization/universidade-...","[/, /organization, /organization/universidade-..."
datastore_active,NaN,True,True
format,CSV,CSV,CSV
formato,CSV,CSV,CSV


In [55]:
df_datagov.format.value_counts()

CSV    55
PDF     3
Name: format, dtype: int64

In [56]:
df_datagov.formato.value_counts()

CSV    55
PDF     3
Name: formato, dtype: int64

In [57]:
FN = lambda ext='json' : './dadosgov_{}_{}_query.{}'.format(alias,query.replace('+','_'), ext)
 
df_datagov.to_json(FN('json'))
df_datagov.to_csv(FN('csv'))
df_datagov.to_excel(FN('xls'))
for fn in ['json', 'csv', 'xls']:
    google.colab.files.download(FN(fn))

<ipython-input-57-cd87016e7687>:5: FutureWarning: As the xlwt package is no longer maintained, the xlwt engine will be removed in a future version of pandas. This is the only engine in pandas that supports writing in the xls format. Install openpyxl and write to an xlsx file instead. You can set the option io.excel.xls.writer to 'xlwt' to silence this warning. While this option is deprecated and will also raise a warning, it can be globally set and the warning suppressed.
  df_datagov.to_excel(FN('xls'))



ModuleNotFoundError: No module named 'xlwt'

# Leitura do banco de dados

In [58]:
df_discentes = df_datagov[df_datagov.dataset_key == 'discentes']
df_discentes = df_discentes[df_discentes.format == 'CSV']

In [59]:
DF_DISCENTES = []
for u,url in enumerate(df_discentes.resource_url_out.values):
    print(u,url)
    _dataframe = pandas.read_csv(url,encoding='UTF-8',sep=';')
    #_dataframe = pandas.read_csv(url,encoding='latin1',sep=';')
    DF_DISCENTES.append(_dataframe)
    

0 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/a55aef81-e094-4267-8643-f283524e3dd7/download/discentes-2019.csv
1 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/146b749b-b9d0-49b2-b114-ac6cc82a4051/download/discentes-2018.csv
2 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/dc732572-a51a-4d4a-a39d-2db37cbe5382/download/discentes-2017.csv
3 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/7d2fa5b3-743f-465f-8450-91719b34a002/download/discentes-2016.csv
4 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/e2b5b843-4f58-497e-8979-44daf8df8f94/download/discentes-2015.csv
5 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/6c23a430-9a7c-4d0f-9602-1d5d97d40e6a/download/discentes-2014.csv
6 https://dados.ufrn.br/dataset/554c2d41-cfce-4278-93c6-eb9aa49c5d16/resource/dba208c2-822f-4e26-adc3-b61d4cb110b6/download/discentes-2013.csv

In [60]:
DF_DISCENTES = pandas.concat(DF_DISCENTES)

In [81]:
NAME = []
for name in DF_DISCENTES[DF_DISCENTES.sexo == 'F']['nome_discente'].values:
    NAME.append(name.split()[0])

In [86]:
print(*sorted(set(filter(lambda x: re.findall('^U',x),set(NAME)))),sep='\n')

UAIANA
UBERLANDIA
UBERLANY
UBERLÂNDIA
UBIKARLA
UBIRACIRA
UBIRANDILMA
UBIRANEIDE
UBIRANILDA
UDICELI
UDIMARA
UEDINA
UELIANIA
UELIDA
UELMA
UENIA
UERLEIDE
UEZIA
UIACI
UIACY
UIARA
UIATANIRA
UIHARA
UILARA
UILIANE
UILIETE
UILLIANE
UILMA
UILSA
UIONARA
UIRANEIDE
UIRATÂNIA
ULANA
ULIANA
ULIANE
ULIANY
ULICÉLIA
ULISANDRA
ULISDETE
ULISMAR
ULISSEIA
ULLA
ULLI
ULLIKENIA
ULLY
ULRIKE
ULYANA
ULYSCLEY
ULYVÂNIA
UMARILANDIA
UMBELINA
UOTANA
URANEIDE
URANIA
URRACA
URSULA
UTA
UYARA
UZIELMA
UZINETE
UÉLIDA


In [123]:
DF_DISCENTES[DF_DISCENTES.nome_discente.transform(lambda nome: re.findall('^JOSÉ HENRIQUE',nome)!=[] )]

,matricula,nome_discente,sexo,ano_ingresso,periodo_ingresso,forma_ingresso,tipo_discente,status,sigla_nivel_ensino,nivel_ensino,id_curso,nome_curso,modalidade_educacao,id_unidade,nome_unidade,id_unidade_gestora,nome_unidade_gestora
7846,2.019302e+10,JOSÉ HENRIQUE MONTEIRO KREIMER FILHO,M,2019,1.0,PROCESSO SELETIVO,REGULAR,CANCELADO,T,TÉCNICO,4970.0,TÉCNICO DE MÚSICA (INSTRUMENTO),PRESENCIAL,284.0,ESCOLA DE MÚSICA,605.0,UNIVERSIDADE FEDERAL DO RIO GRANDE DO NORTE
7847,2.019300e+10,JOSÉ HENRIQUE MOREIRA PINHEIRO,M,2019,1.0,PROCESSO SELETIVO,REGULAR,ATIVO,T,TÉCNICO,96054058.0,CURSO TÉCNICO DA METRÓPOLE DIGITAL,SEMI-PRESENCIAL,6069.0,INSTITUTO METROPOLE DIGITAL,605.0,UNIVERSIDADE FEDERAL DO RIO GRANDE DO NORTE
10161,2.018203e+10,JOSÉ HENRIQUE BRANDINI NÉSPOLI,M,2018,2.0,PROCESSO SELETIVO,REGULAR,ATIVO,L,LATO SENSU,128492008.0,ESPECIALIZAÇÃO EM PRECEPTORIA EM SAÚDE,A DISTÂNCIA,205.0,ESCOLA DE SAÚDE,605.0,UNIVERSIDADE FEDERAL DO RIO GRANDE DO NORTE
10162,2.018000e+10,JOSÉ HENRIQUE DE SOUZA MELO,M,2018,1.0,REOCUPAÇÃO DE VAGAS RESIDUAIS,REGULAR,ATIVO,G,GRADUAÇÃO,111635075.0,QUÍMICA,PRESENCIAL,439.0,CENTRO DE CIÊNCIAS EXATAS E DA TERRA,439.0,CENTRO DE CIÊNCIAS EXATAS E DA TERRA
10163,2.018302e+10,JOSÉ HENRIQUE MOREIRA DA COSTA,M,2018,1.0,PROCESSO SELETIVO,REGULAR,CONCLUÍDO,T,TÉCNICO,94763016.0,TÉCNICO EM AGENTE COMUNITÁRIO DE SAÚDE,PRESENCIAL,205.0,ESCOLA DE SAÚDE,605.0,UNIVERSIDADE FEDERAL DO RIO GRANDE DO NORTE
10164,2.018006e+10,JOSÉ HENRIQUE TARGINO DIAS GÓIS,M,2018,1.0,SiSU,REGULAR,ATIVO,G,GRADUAÇÃO,92127264.0,TECNOLOGIA DA INFORMAÇÃO,PRESENCIAL,6069.0,INSTITUTO METROPOLE DIGITAL,605.0,UNIVERSIDADE FEDERAL DO RIO GRANDE DO NORTE
11532,2.017200e+10,JOSÉ HENRIQUE DE SOUZA NETO,M,2017,1.0,PROCESSO SELETIVO,REGULAR,CANCELADO,L,LATO SENSU,117029625.0,"ESPECIALIZAÇÃO MBA EM CONTROLADORIA, FINANÇAS ...",PRESENCIAL,163.0,DEPARTAMENTO DE CIÊNCIAS CONTÁBEIS - DCC,443.0,CENTRO DE CIÊNCIAS SOCIAIS APLICADAS
9282,2.016101e+09,JOSÉ HENRIQUE TARGINO DIAS GÓIS,M,2016,1.0,SELEÇÃO DE PÓS-GRADUAÇÃO,REGULAR,CONCLUÍDO,D,DOUTORADO,17565631.0,DOUTORADO EM NEUROCIÊNCIAS,PRESENCIAL,5207.0,PROGRAMA DE PÓS-GRADUAÇÃO EM NEUROCIÊNCIAS,605.0,UNIVERSIDADE FEDERAL DO RIO GRANDE DO NORTE
9851,2.015012e+10,JOSÉ HENRIQUE ALVES FERREIRA,M,2015,1.0,SiSU,REGULAR,CANCELADO,G,GRADUAÇÃO,111635050.0,FÍSICA,PRESENCIAL,439.0,CENTRO DE CIÊNCIAS EXATAS E DA TERRA,439.0,CENTRO DE CIÊNCIAS EXATAS E DA TERRA
9853,2.015201e+10,JOSÉ HENRIQUE VIEIRA DA CRUZ,M,2015,1.0,PROCESSO SELETIVO,REGULAR,CONCLUÍDO,L,LATO SENSU,106806058.0,ESPECIALIZAÇÃO EM GEOPROCESSAMENTO E CARTOGRAF...,PRESENCIAL,142.0,DEPARTAMENTO DE GEOGRAFIA/CCHLA,442.0,"CENTRO DE CIÊNCIAS HUMANAS, LETRAS E ARTES"


In [130]:
import string

In [131]:
DF_DISCENTES.nome_discente.transform(lambda n: n.split())

0              [ABDENOR, BEZERRA, DOS, SANTOS]
1        [ABDIAS, MONTEIRO, DE, ANDRADE, MELO]
2           [ABDIAS, SABINO, RODRIGUES, FILHO]
3           [ABEL, GOMES, DE, OLIVEIRA, FILHO]
4            [ABI, AMANA, DE, AQUINO, BEZERRA]
                         ...                  
5301             [ZILENE, TAVARES, DE, CASTRO]
5302       [ZILKARLA, DOS, SANTOS, DE, ARAUJO]
5303      [ZOETE, FERREIRA, NEVES, DE, ARAUJO]
5304     [ZUILSON, AUGUSTO, CARLOS, DA, SILVA]
5305    [ZWINGLA, TEONIA, ALVES, DE, MEDEIROS]
Name: nome_discente, Length: 329376, dtype: object